In [ ]:

from dotenv import load_dotenv
load_dotenv()

import json
import os
import sys
from litellm import completion
from typing import List, Dict

TOOLS = [{
    "type": "function",
    "function": {
        "name": "gerar_falas",
        "description": "Gera as falas de pré-luta ou pós-luta entre dois personagens do mundo do jogo Terceira Lua",
        "parameters": {
            "type": "object",
            "properties": {
                "fala_a": {"type": "string", "description": "Fala curta do personagem A"},
                "fala_b": {"type": "string", "description": "Fala curta do personagem B"}
            },
            "required": ["fala_a", "fala_b"]
        }
    }
}]

def gerar_dialogo(personagem_a, personagem_b, momento="pre_luta", vencedor=None):
    
 #   personagem_a / personagem_b: dicts com {"nome", "personalidade", "tom"}
 #   momento: "pre_luta" ou "pos_luta"
  #  vencedor: nome do vencedor, só usado se momento == "pos_luta"
    
    contexto = f"""
    Personagem A: {personagem_a['nome']} — {personagem_a['historia']}, tom: {personagem_a['tom']}
    Personagem B: {personagem_b['nome']} — {personagem_b['historia']}, tom: {personagem_b['tom']}
    Momento: {momento}
    """
    if momento == "pos_luta":
        contexto += f"\nVencedor: {vencedor}. A fala do vencedor deve provocar; a do perdedor deve reagir à derrota."

    response = completion(
        model="groq/llama-3.1-8b-instant",  # modelo pequeno = rápido e barato (Model Efficiency do MATE)
        messages=[
            {"role": "system", "content": "Você escreve falas curtas (máximo: 12 palavras) de personagens de jogo de luta, mantendo o tom de cada um. Responda só pela função."},
            {"role": "user", "content": contexto}
        ],
        tools=TOOLS,
        api_key=os.getenv("GROQ_API_KEY"),
        timeout=3  # não travar a tela de loading
    )

    tool_call = response.choices[0].message.tool_calls[0]
    return json.loads(tool_call.function.arguments)  # {"fala_a": "...", "fala_b": "..."}